# SEIRS-SEI Malaria Model — First Week Parameter Fitting

Fitting parameters b1, b2, H0, k, phi to match I_H from the model to the data (daily) for the first week of 2017.

The goal is to find parameters that make I_H decrease in the first week, matching the observed data trend.

**Parameters to fit:**
- `b1`: probability of transmission from infected human to mosquito (∈ [0, 1])
- `b2`: probability of transmission from infected mosquito to human (∈ [0, 1])
- `H0`: humidity threshold for mosquito survival sigmoid
- `k`: steepness of the humidity sigmoid
- `phi`: baseline mosquito survival probability (∈ [0, 1])

**Fixed**: All other parameters from the shared model, including initial conditions and M'.

In [ ]:
import sys
sys.path.append("..")

from ode_models.seirs_sei_ode_shared import *
from scipy.optimize import minimize
%matplotlib inline

warnings.filterwarnings("ignore")

In [ ]:
#### Configuration
FIRST_WEEK_DAYS = 7  # Number of days to fit
USE_SMOOTH_CLIMATE = True

In [ ]:
#### Prepare 2017 first-week data
year = 2017
N_year = round(pop_by_year[year])

start = f"{year}-01-01"
end_first_week = f"{year}-01-07"

# Climate data for the first week
year_climate = climate_data[
    (climate_data['date'] >= start) & (climate_data['date'] <= end_first_week)
].reset_index(drop=True)

if USE_SMOOTH_CLIMATE:
    year_climate['temp_med_smooth'] = year_climate['temp_med'].rolling(SMOOTH_WINDOW, min_periods=1, center=False).mean()
    year_climate['precip_med_smooth'] = year_climate['precip_med'].rolling(SMOOTH_WINDOW, min_periods=1, center=False).mean()
    year_climate['umid_min_smooth'] = year_climate['umid_min'].rolling(SMOOTH_WINDOW, min_periods=1, center=False).mean()

num_days = len(year_climate)
print(f"First week of {year}: {num_days} days ({start} to {end_first_week})")
print(f"Climate columns: {list(year_climate.columns)}")

In [ ]:
#### Observed data for first week
rural_cases_first_week = rural_cases_df[
    (rural_cases_df['date'] >= start) & (rural_cases_df['date'] <= end_first_week)
].reset_index(drop=True)

observed_IH = rural_cases_first_week['active_total'].values
observed_dates = rural_cases_first_week['date'].values

print("Observed I_H (active_total) for first week:")
for i, (d, val) in enumerate(zip(observed_dates, observed_IH)):
    print(f"  Day {i+1} ({pd.Timestamp(d).strftime('%Y-%m-%d')}): {val:.1f}")

In [ ]:
#### Simulation function for the first week
def simulate_first_week(b1_val, b2_val, H0_val, k_val, phi_val, return_full=False):
    """
    Run the ODE for the first week with given parameters.
    Returns daily I_H values.
    """
    def ode_func(t, z):
        return seirs_sei_ode(
            t, z, N_year, year_climate,
            T_prime, B_E, p_ME, p_ML, p_MP, tau_E, tau_P,
            c1, c2, D1, b1_val, b2_val, A, B, C, DD,
            Tmin, optimal_temp_plasm, critical_max_temp_plasm,
            optimal_temp_anop, critical_max_temp_anop, critical_min_temp_anop,
            gamma, R_L, M_prime, tau_H, omega,
            M_min=M_min, permanent_factor=permanent_factor,
            use_smooth_climate=USE_SMOOTH_CLIMATE,
            H0=H0_val, k=k_val, phi=phi_val,
        )

    # Solve with daily output points
    t_eval = np.linspace(0, num_days, num_days * 10)
    sol = solve_ivp(ode_func, [0, num_days], initial_state_2017,
                    t_eval=t_eval, method='LSODA')

    if not sol.success:
        return None

    # Interpolate to daily resolution matching the climate dates
    t_daily = np.linspace(0, num_days, len(year_climate))
    IH_daily = interp1d(sol.t, sol.y[2])(t_daily)

    if return_full:
        return sol, IH_daily
    return IH_daily

In [ ]:
#### Baseline (default parameters)
print("Default parameters:")
print(f"  b1 = {b1}")
print(f"  b2 = {b2}")
print(f"  H0 = 58.0  (default)")
print(f"  k  = 0.25  (default)")
print(f"  phi = 0.05 (default)")

IH_baseline = simulate_first_week(b1, b2, 58.0, 0.25, 0.05)
if IH_baseline is not None:
    baseline_mse = np.mean((IH_baseline - observed_IH) ** 2)
    print(f"\nBaseline MSE: {baseline_mse:,.2f}")
    print("Baseline I_H vs Observed:")
    for i, (mod, obs) in enumerate(zip(IH_baseline, observed_IH)):
        print(f"  Day {i+1}: Model={mod:.1f}, Observed={obs:.1f}")
else:
    print("Baseline simulation failed!")

In [ ]:
#### Objective function
# Params order: [b1, b2, H0, k, phi]

def objective(params):
    b1_val, b2_val, H0_val, k_val, phi_val = params

    # Enforce bounds for probabilities
    if not (0 <= b1_val <= 1):
        return 1e15
    if not (0 <= b2_val <= 1):
        return 1e15
    if not (0 <= phi_val <= 1):
        return 1e15
    if H0_val < 0 or H0_val > 100:
        return 1e15
    if k_val < 0 or k_val > 10:
        return 1e15

    IH = simulate_first_week(b1_val, b2_val, H0_val, k_val, phi_val)
    if IH is None:
        return 1e15

    mse = np.mean((IH - observed_IH) ** 2)
    return mse

In [ ]:
#### Parameter bounds
# b1, b2: probabilities ∈ [0, 1]
# H0: humidity threshold (reasonable range for relative humidity)
# k: sigmoid steepness
# phi: baseline probability ∈ [0, 1]

bounds = [
    (0.0, 1.0),    # b1
    (0.0, 1.0),    # b2
    (0.0, 100.0),  # H0
    (0.0, 10.0),   # k
    (0.0, 1.0),    # phi
]

initial_guess = [b1, b2, 58.0, 0.25, 0.05]
print(f"Initial guess: b1={initial_guess[0]}, b2={initial_guess[1]}, H0={initial_guess[2]}, k={initial_guess[3]}, phi={initial_guess[4]}")

In [ ]:
#### Optimization with L-BFGS-B (supports bounds)
print("Optimizing with L-BFGS-B...")

result = minimize(
    objective,
    initial_guess,
    method='L-BFGS-B',
    bounds=bounds,
    options={'maxiter': 200, 'ftol': 1e-12, 'gtol': 1e-8}
)

print(f"\nOptimization {'succeeded' if result.success else 'FAILED'}")
print(f"Optimizer message: {result.message}")
print(f"Final MSE: {result.fun:,.2f}")
print(f"\nOptimized parameters:")
print(f"  b1  = {result.x[0]:.6f}")
print(f"  b2  = {result.x[1]:.6f}")
print(f"  H0  = {result.x[2]:.6f}")
print(f"  k   = {result.x[3]:.6f}")
print(f"  phi = {result.x[4]:.6f}")

In [ ]:
#### Try multiple random starts to avoid local minima
print("\n" + "="*60)
print("Multi-start optimization with random initial guesses...")
print("="*60)

np.random.seed(42)
n_restarts = 20

best_result = result
results = [result]

for i in range(n_restarts):
    # Generate random initial guess within bounds
    x0 = [
        np.random.uniform(0, 1),       # b1
        np.random.uniform(0, 1),       # b2
        np.random.uniform(0, 100),     # H0
        np.random.uniform(0, 5),       # k
        np.random.uniform(0, 1),       # phi
    ]

    r = minimize(
        objective,
        x0,
        method='L-BFGS-B',
        bounds=bounds,
        options={'maxiter': 200, 'ftol': 1e-12, 'gtol': 1e-8}
    )
    results.append(r)

    if r.success and r.fun < best_result.fun:
        best_result = r
        print(f"  Restart {i+1:2d}: NEW BEST MSE={r.fun:,.2f}  "
              f"b1={r.x[0]:.4f} b2={r.x[1]:.4f} H0={r.x[2]:.1f} k={r.x[3]:.4f} phi={r.x[4]:.4f}")
    elif r.success:
        pass  # silently report only improvements
    else:
        print(f"  Restart {i+1:2d}: failed ({r.message})")

print(f"\nBest result from {n_restarts+1} runs:")
print(f"  MSE = {best_result.fun:,.2f}")
print(f"  b1  = {best_result.x[0]:.6f}")
print(f"  b2  = {best_result.x[1]:.6f}")
print(f"  H0  = {best_result.x[2]:.6f}")
print(f"  k   = {best_result.x[3]:.6f}")
print(f"  phi = {best_result.x[4]:.6f}")

In [ ]:
#### Evaluate optimized parameters
opt_b1, opt_b2, opt_H0, opt_k, opt_phi = best_result.x

sol_opt, IH_opt = simulate_first_week(opt_b1, opt_b2, opt_H0, opt_k, opt_phi, return_full=True)

print("Optimized model I_H vs Observed (first week):")
print(f"  {'Day':<5} {'Date':<12} {'Model I_H':<12} {'Observed':<12} {'Diff':<12}")
print(f"  " + "-"*50)
for i in range(len(observed_IH)):
    diff = IH_opt[i] - observed_IH[i]
    print(f"  {i+1:<5} {pd.Timestamp(observed_dates[i]).strftime('%Y-%m-%d'):<12} "
          f"{IH_opt[i]:<12.1f} {observed_IH[i]:<12.1f} {diff:<+12.1f}")

opt_mse = np.mean((IH_opt - observed_IH) ** 2)
print(f"\nOptimized MSE: {opt_mse:,.2f}")
print(f"Baseline MSE:  {baseline_mse:,.2f}")
print(f"Improvement:   {(1 - opt_mse/baseline_mse)*100:.1f}%")

In [ ]:
#### Plot: Baseline vs Optimized vs Observed (First Week)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

dates_str = [pd.Timestamp(d).strftime('%m-%d') for d in observed_dates]
x = np.arange(len(observed_IH))

# Panel 1: I_H comparison
ax = axes[0]
ax.plot(x, IH_baseline, 'b-o', label=f'Baseline (MSE={baseline_mse:,.0f})', markersize=6)
ax.plot(x, IH_opt, 'g-s', label=f'Optimized (MSE={opt_mse:,.0f})', markersize=6)
ax.plot(x, observed_IH, 'r-^', label='Observed', markersize=6)
ax.set_xticks(x)
ax.set_xticklabels(dates_str)
ax.set_xlabel('Date (Jan 2017)')
ax.set_ylabel('Infectious Humans (I_H)')
ax.set_title('First Week: Model vs Data')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: Full ODE trajectories over first week
ax = axes[1]

# Re-run baseline to get full trajectory
sol_base, IH_base_full = simulate_first_week(b1, b2, 58.0, 0.25, 0.05, return_full=True)

t_fine = np.linspace(0, num_days, num_days*100)
IH_base_fine = interp1d(sol_base.t, sol_base.y[2])(t_fine)
IH_opt_fine = interp1d(sol_opt.t, sol_opt.y[2])(t_fine)

ax.plot(t_fine, IH_base_fine, 'b-', alpha=0.5, label='Baseline')
ax.plot(t_fine, IH_opt_fine, 'g-', alpha=0.5, label='Optimized')
ax.scatter(x, observed_IH, color='red', zorder=5, label='Observed')
ax.set_xlabel('Time (days)')
ax.set_ylabel('I_H')
ax.set_title('Full ODE Trajectory (First Week)')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: Parameter comparison
ax = axes[2]
ax.axis('off')
param_names = ['b1', 'b2', 'H0', 'k', 'phi']
default_vals = [b1, b2, 58.0, 0.25, 0.05]
opt_vals = [opt_b1, opt_b2, opt_H0, opt_k, opt_phi]

table_data = []
table_data.append(['Parameter', 'Default', 'Optimized', 'Change'])
for i, name in enumerate(param_names):
    change_pct = ((opt_vals[i] - default_vals[i]) / max(abs(default_vals[i]), 1e-10)) * 100
    table_data.append([name, f'{default_vals[i]:.4f}', f'{opt_vals[i]:.4f}', f'{change_pct:+.1f}%'])
table_data.append(['MSE', f'{baseline_mse:,.0f}', f'{opt_mse:,.0f}', f'{(1-opt_mse/baseline_mse)*100:+.1f}%'])

col_widths = [max(len(row[j]) for row in table_data) + 2 for j in range(4)]
for row_idx, row in enumerate(table_data):
    line = '  '.join(cell.ljust(col_widths[i]) for i, cell in enumerate(row))
    ax.text(0.1, 0.95 - row_idx*0.08, line, fontsize=11, family='monospace',
            fontweight='bold' if row_idx == 0 else 'normal',
            color='blue' if row_idx == len(table_data)-1 else 'black')

ax.set_title('Parameter Comparison', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
#### Extended validation: Run full year with optimized params to see behavior
print("="*60)
print("Extended simulation: Full 2017 with optimized parameters")
print("="*60)

years = [2017]
all_results = []
current_state = initial_state_2017.copy()

for yr in years:
    N_yr = round(pop_by_year[yr])
    start_yr = f"{yr}-01-01"
    end_yr = f"{yr}-12-31"

    yr_climate = climate_data[
        (climate_data['date'] >= start_yr) & (climate_data['date'] <= end_yr)
    ].reset_index(drop=True)

    if USE_SMOOTH_CLIMATE:
        yr_climate['temp_med_smooth'] = yr_climate['temp_med'].rolling(SMOOTH_WINDOW, min_periods=1, center=False).mean()
        yr_climate['precip_med_smooth'] = yr_climate['precip_med'].rolling(SMOOTH_WINDOW, min_periods=1, center=False).mean()
        yr_climate['umid_min_smooth'] = yr_climate['umid_min'].rolling(SMOOTH_WINDOW, min_periods=1, center=False).mean()

    num_days_yr = len(yr_climate)

    def ode_func_yr(t, z):
        return seirs_sei_ode(
            t, z, N_yr, yr_climate,
            T_prime, B_E, p_ME, p_ML, p_MP, tau_E, tau_P,
            c1, c2, D1, opt_b1, opt_b2, A, B, C, DD,
            Tmin, optimal_temp_plasm, critical_max_temp_plasm,
            optimal_temp_anop, critical_max_temp_anop, critical_min_temp_anop,
            gamma, R_L, M_prime, tau_H, omega,
            M_min=M_min, permanent_factor=permanent_factor,
            use_smooth_climate=USE_SMOOTH_CLIMATE,
            H0=opt_H0, k=opt_k, phi=opt_phi,
        )

    sol = solve_ivp(ode_func_yr, [0, num_days_yr], current_state,
                    t_eval=np.linspace(0, num_days_yr, num_days_yr*10),
                    method='LSODA')

    all_results.append({'year': yr, 'N': N_yr, 'sol': sol, 'climate': yr_climate})
    current_state = sol.y[:, -1].copy()
    print(f"Year {yr}: Final I_H = {round(current_state[2])}, Final I_M = {round(current_state[6])}")

# Interpolate to daily
combined_df = pd.DataFrame()
for res in all_results:
    yr = res['year']
    sol = res['sol']
    climate = res['climate']
    t_interp = np.linspace(0, sol.t[-1], len(climate))
    interp = [interp1d(sol.t, sol.y[i])(t_interp) for i in range(7)]
    df = pd.DataFrame({
        'date': pd.to_datetime(climate['date']).values,
        'S_H': interp[0], 'E_H': interp[1], 'I_H': interp[2], 'R_H': interp[3],
        'S_M': interp[4], 'E_M': interp[5], 'I_M': interp[6], 'N': res['N']
    })
    combined_df = pd.concat([combined_df, df]).reset_index(drop=True)

In [ ]:
#### Plot: Full year comparison
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

ax = axes[0]
ax.plot(combined_df['date'], combined_df['I_H'], 'g-', linewidth=2,
        label=f'Optimized Model (MSE={opt_mse:,.0f})', alpha=0.8)
ax.plot(rural_cases_df['date'], rural_cases_df['active_total'], 'r-', linewidth=1.5,
        label='Observed Active Rural Cases', alpha=0.7)
ax.set_title('Model vs Data: Infectious Humans — Full 2017 (Optimized Params)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
# Zoom to first month
mask_mod = combined_df['date'] < pd.to_datetime('2017-02-01')
mask_obs = rural_cases_df['date'] < pd.to_datetime('2017-02-01')
ax.plot(combined_df['date'][mask_mod], combined_df['I_H'][mask_mod], 'g-', linewidth=2, label='Optimized Model')
ax.plot(rural_cases_df['date'][mask_obs], rural_cases_df['active_total'][mask_obs],
        'ro-', markersize=4, linewidth=1.5, label='Observed')
ax.set_title('Zoom: First Month of 2017')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
#### Parameter sensitivity (perturb each parameter individually)
print("="*60)
print("Parameter Sensitivity Analysis")
print("="*60)

perturbations = [0.5, 0.8, 0.9, 1.1, 1.2, 1.5]  # multipliers
param_names = ['b1', 'b2', 'H0', 'k', 'phi']
opt_values = [opt_b1, opt_b2, opt_H0, opt_k, opt_phi]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes_flat = axes.flatten()

for p_idx, (pname, pval) in enumerate(zip(param_names, opt_values)):
    ax = axes_flat[p_idx]
    mspe = []
    for mult in perturbations:
        test_vals = opt_values.copy()
        test_vals[p_idx] = pval * mult

        # Clamp to bounds
        if pname in ['b1', 'b2', 'phi']:
            test_vals[p_idx] = max(0, min(1, test_vals[p_idx]))
        elif pname == 'H0':
            test_vals[p_idx] = max(0, min(100, test_vals[p_idx]))
        elif pname == 'k':
            test_vals[p_idx] = max(0, test_vals[p_idx])

        ih = simulate_first_week(*test_vals)
        if ih is not None:
            mse = np.mean((ih - observed_IH) ** 2)
        else:
            mse = np.nan
        mspe.append(mse)

    ax.semilogy(perturbations, mspe, 'o-', linewidth=2)
    ax.axvline(1.0, color='red', linestyle='--', alpha=0.5, label='Optimal')
    ax.axhline(opt_mse, color='gray', linestyle=':', alpha=0.5, label=f'Opt MSE={opt_mse:.0f}')
    ax.set_xlabel(f'{pname} multiplier')
    ax.set_ylabel('MSE')
    ax.set_title(f'Sensitivity: {pname} (optimal={pval:.4f})')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes_flat[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
#### Summary
print("="*60)
print("FITTING SUMMARY")
print("="*60)
print(f"\nFitting period: {start} to {end_first_week}")
print(f"Parameters fitted: b1, b2, H0, k, phi")
print(f"Multi-start runs: {n_restarts + 1}")
print(f"\n{'Parameter':<12} {'Default':<12} {'Optimized':<12} {'Bound'}")
print(f"{'---------':<12} {'-------':<12} {'---------':<12} {'-----'}")
for name, default, opt, bound in zip(
    ['b1', 'b2', 'H0', 'k', 'phi'],
    [b1, b2, 58.0, 0.25, 0.05],
    [opt_b1, opt_b2, opt_H0, opt_k, opt_phi],
    ['[0, 1]', '[0, 1]', '[0, 100]', '[0, 10]', '[0, 1]']
):
    print(f"{name:<12} {default:<12.4f} {opt:<12.4f} {bound}")
print(f"\nMSE:  {baseline_mse:>12,.2f}  →  {opt_mse:>12,.2f}")
print(f"RMSE: {np.sqrt(baseline_mse):>12,.2f}  →  {np.sqrt(opt_mse):>12,.2f}")